# 6. Análisis exploratorio
Ejecute primero el notebook 01. Los conteos de víctimas, los conteos territoriales y los incidentes de Línea 123 se presentan por separado. Las tablas y figuras quedan en `data/processed/`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# En Colab, sitúe el directorio de trabajo en la raíz del repositorio.
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
raw = root / 'data' / 'raw'
processed = root / 'data' / 'processed'
processed.mkdir(parents=True, exist_ok=True)

def input_path(name):
    for folder in [root / 'DataSet', raw]:
        path = folder / name
        if path.is_file():
            return path
    return raw / name

import matplotlib.pyplot as plt

def save_figure(ax, filename):
    ax.figure.tight_layout()
    ax.figure.savefig(processed / filename, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(ax.figure)

In [ ]:
daily = pd.read_csv(processed / 'hurto_diario.csv', parse_dates=['fecha']).set_index('fecha')
counts = daily['cantidad']
display(counts.describe())
ax = counts.plot(figsize=(12, 4), lw=.65, title='Hurto a personas registrado en Bogotá')
ax.set(xlabel='Fecha', ylabel='Víctimas registradas')
save_figure(ax, 'serie_diaria.png')

monthly = counts.resample('MS').agg(['sum', 'mean', 'size'])
monthly.columns = ['victimas', 'promedio_diario', 'dias']
monthly.to_csv(processed / 'hurto_mensual.csv')
display(monthly.nlargest(3, 'victimas'))
display(monthly.nsmallest(3, 'victimas'))
ax = monthly['victimas'].plot(figsize=(12, 4), title='Víctimas registradas por mes')
ax.set(xlabel='Mes', ylabel='Víctimas registradas')
save_figure(ax, 'hurto_mensual.png')

weekday = counts.groupby(daily.index.dayofweek).agg(['size', 'mean', 'median', 'sum'])
weekday.index = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
weekday.columns = ['dias', 'promedio', 'mediana', 'victimas']
weekday.to_csv(processed / 'hurto_por_dia_semana.csv')
display(weekday.round(2))
ax = weekday['promedio'].plot.bar(rot=25, color='#247480', title='Promedio de víctimas por día de la semana')
ax.set(ylabel='Víctimas registradas por día')
save_figure(ax, 'hurto_por_dia_semana.png')

## Conteos anuales por localidad
La categoría Sin Localización forma parte de los totales distritales. Su geometría no se usa para atribuirle una ubicación.

In [ ]:
import numpy as np

path = input_path('localidades.geojson')
if path.is_file():
    import geopandas as gpd

    areas = gpd.read_file(path)
    fields = {f'CMHP{year % 100:02}CONT': year for year in range(2018, 2026)}
    required = ['CMNOMLOCAL', *fields]
    missing = set(required) - set(areas.columns)
    if missing:
        raise ValueError(f'Campos territoriales ausentes: {sorted(missing)}')
    wide = areas[required].rename(columns={'CMNOMLOCAL': 'localidad', **fields})
    wide['localidad'] = wide['localidad'].str.strip()
    if wide['localidad'].isna().any() or wide['localidad'].eq('').any() or wide['localidad'].duplicated().any():
        raise ValueError('Nombres de localidad faltantes o duplicados.')
    unlocated = wide['localidad'].str.casefold().eq('sin localización')
    wide.loc[unlocated, 'localidad'] = 'Sin Localización'
    annual = wide.melt(id_vars='localidad', var_name='anio', value_name='conteo')
    annual['conteo'] = pd.to_numeric(annual['conteo'], errors='coerce')
    invalid = ~np.isfinite(annual['conteo']) | annual['conteo'].lt(0) | annual['conteo'].mod(1).ne(0)
    print('Filas de la capa:', len(areas), '| Filas localidad–año:', len(annual),
          '| Conteos inválidos:', invalid.sum(), '| Filas Sin Localización:', unlocated.sum())
    if invalid.any() or unlocated.sum() != 1:
        raise ValueError('Revisar conteos o categoría Sin Localización.')
    annual.to_csv(processed / 'hurto_localidad_anio.csv', index=False)
    totals = annual.groupby('anio')['conteo'].sum().rename('total_distrital').to_frame()
    totals['sin_localizacion'] = annual.loc[annual['localidad'].eq('Sin Localización')].set_index('anio')['conteo']
    display(totals)
    display(annual.loc[annual['anio'].eq(2025) & annual['localidad'].ne('Sin Localización')].nlargest(5, 'conteo'))
    ax = totals['total_distrital'].plot.bar(rot=0, title='Conteos anuales en la capa distrital')
    ax.set(xlabel='Año', ylabel='Conteo de hurto a personas')
    save_figure(ax, 'hurto_territorial_anual.png')

    polygons = areas.loc[~unlocated].copy()
    usable = polygons.geom_type.isin(['Polygon', 'MultiPolygon']) & polygons.is_valid & ~polygons.is_empty
    print('Localidades con polígonos válidos:', usable.sum(), 'de', len(polygons))
    if usable.any():
        polygons = polygons.loc[usable].assign(conteo_2025=wide.loc[usable.index[usable], 2025])
        ax = polygons.to_crs('EPSG:3116').plot(
            column='conteo_2025', cmap='YlOrRd', edgecolor='#444444', linewidth=.5,
            legend=True, figsize=(7, 8), legend_kwds={'label': 'Conteo registrado'}
        )
        ax.set_title('Hurto a personas por localidad · 2025')
        ax.set_axis_off()
        save_figure(ax, 'hurto_localidades_2025.png')
else:
    print('Pendiente: descargar la capa territorial según data/README.md.')

## Línea 123
Exploración complementaria por mes, tipo y localidad. Se suman los incidentes del campo CANT_INCIDENTES. El filtro textual de hurto se audita mediante el listado de categorías.

In [ ]:
path = input_path('linea123.csv')
if path.is_file():
    reports = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype='string')
    fields = {'ID', 'ANIO', 'MES', 'TIPO_INCIDENTE', 'TIPO_DETALLE', 'COD_LOCALIDAD', 'LOCALIDAD', 'CANT_INCIDENTES'}
    if fields - set(reports.columns):
        raise ValueError(f'Campos de Línea 123 ausentes: {sorted(fields - set(reports.columns))}')
    display(pd.DataFrame({'tipo_leido': reports.dtypes.astype(str), 'nulos': reports.isna().sum()}))
    print('ID repetidos:', reports['ID'].duplicated().sum(),
          '| Duplicados exactos:', reports.duplicated().sum())

    reports['ANIO'] = pd.to_numeric(reports['ANIO'], errors='coerce')
    reports['MES'] = pd.to_numeric(reports['MES'], errors='coerce')
    reports['CANT_INCIDENTES'] = pd.to_numeric(reports['CANT_INCIDENTES'], errors='coerce')
    amount = reports['CANT_INCIDENTES']
    invalid = ~np.isfinite(amount) | amount.lt(0) | amount.mod(1).ne(0)
    bad_dates = (
        reports['ANIO'].isna() | reports['ANIO'].mod(1).ne(0)
        | ~reports['MES'].between(1, 12) | reports['MES'].mod(1).ne(0)
    )
    print('Filas con periodo inválido:', bad_dates.sum(), '| Conteos inválidos:', invalid.sum())
    if invalid.any() or bad_dates.any():
        raise ValueError('Revisar periodos y conteos de Línea 123 antes de agregar.')

    historical = reports.loc[reports['ANIO'].between(2015, 2025)]
    study = reports.loc[reports['ANIO'].between(2018, 2025)]
    theft = study.loc[study['TIPO_DETALLE'].str.contains(r'\bhurtos?\b', case=False, na=False)]
    volumes = pd.DataFrame(
        [(len(group), group['CANT_INCIDENTES'].sum()) for group in [reports, historical, study, theft]],
        index=['Archivo completo', '2015–2025, todos', '2018–2025, todos', '2018–2025, hurto'],
        columns=['filas', 'suma_incidentes']
    )
    volumes['referencia_filas'] = pd.Series({'2015–2025, todos': 1044332, '2018–2025, todos': 763074}, dtype='Int64')
    volumes['diferencia_filas'] = volumes['filas'] - volumes['referencia_filas']
    display(volumes)
    volumes.to_csv(processed / 'linea123_volumen.csv')

    categories = theft.groupby(['TIPO_INCIDENTE', 'TIPO_DETALLE'], dropna=False)['CANT_INCIDENTES'].agg(['size', 'sum'])
    display(categories.sort_values('sum', ascending=False))
    keys = ['ANIO', 'MES', 'TIPO_INCIDENTE', 'TIPO_DETALLE', 'COD_LOCALIDAD', 'LOCALIDAD']
    by_zone = theft.groupby(keys, dropna=False)['CANT_INCIDENTES'].agg(filas='size', incidentes='sum')
    by_zone.to_csv(processed / 'linea123_hurto_mes_tipo_localidad.csv')

    if not theft.empty:
        monthly_reports = theft.groupby(['ANIO', 'MES'])['CANT_INCIDENTES'].sum()
        calendar = pd.to_datetime(
            monthly_reports.index.to_frame(index=False).set_axis(['year', 'month'], axis=1).assign(day=1)
        )
        monthly_reports.index = pd.DatetimeIndex(calendar, name='mes')
        monthly_reports.to_csv(processed / 'linea123_hurto_mensual.csv')
        ax = monthly_reports.plot(figsize=(12, 4), title='Incidentes de hurto tramitados por Línea 123')
        ax.set(xlabel='Mes', ylabel='Incidentes tramitados')
        save_figure(ax, 'linea123_hurto_mensual.png')
else:
    print('Pendiente: descargar el CSV de Línea 123 según data/README.md.')